### Packages for the Data Generation and Mopdelling of PD (Probability of Default), LGD (Loss Given Default) and EAD (Exposure at Default)

In [178]:
# Data Management and Processing
import pandas as pd
import numpy as np
import scipy
import random

In [179]:
# Machine Learning and Statistics
import sklearn
import statsmodels.api as sm
import tensorflow as tf

### Data generator

##### Support functions

In [180]:
############# Function to create a profession based on the educational level #########################
def generate_profession(education):
    if education == "high school or lower":
        return random.choices(["LowSkilled", "Unemployed_LowSkilled"], weights=[0.9, 0.1], k=1)[0]
    if education == "ausbildung":
        return random.choices(["MediumSkilled", "Unemployed_MediumSkilled"], weights=[0.9, 0.1], k=1)[0]
    if education in ["bachelor degree", "post graduate degree"]:
        return random.choices(["HighSkilled", "Unemployed_HighSkilled"], weights=[0.9, 0.1], k=1)[0]

In [181]:
############################### Function to generate monthly income and expenditure ##################################                

# Define income parameters for different profession levels and age ranges
income_parameters = {
    ("LowSkilled", "Unemployed_LowSkilled"): {
        (30, 35): {"mean": 1000, "std_dev": 200, "max_income": 2000},
        (36, 40): {"mean": 1200, "std_dev": 200, "max_income": 2400},
        (41, 45): {"mean": 1500, "std_dev": 250, "max_income": 3000},
        (46, 50): {"mean": 1800, "std_dev": 300, "max_income": 3600},
        (51, 55): {"mean": 2000, "std_dev": 350, "max_income": 4000},
        (56, 60): {"mean": 2200, "std_dev": 400, "max_income": 4300},
        (61, 65): {"mean": 2400, "std_dev": 600, "max_income": 4500},
    },
    ("MediumSkilled", "Unemployed_MediumSkilled"): {
        (30, 35): {"mean": 1800, "std_dev": 300, "max_income": 4000},
        (36, 40): {"mean": 2300, "std_dev": 400, "max_income": 5000},
        (41, 45): {"mean": 2600, "std_dev": 500, "max_income": 6000},
        (46, 50): {"mean": 3000, "std_dev": 600, "max_income": 7000},
        (51, 55): {"mean": 3500, "std_dev": 800, "max_income": 8000},
        (56, 60): {"mean": 4000, "std_dev": 800, "max_income": 9000},
        (61, 65): {"mean": 5000, "std_dev": 1000, "max_income": 10000},
    },
    ("HighSkilled", "Unemployed_HighSkilled"): {
        (30, 35): {"mean": 3000, "std_dev": 500, "max_income": 10000},
        (36, 40): {"mean": 4500, "std_dev": 700, "max_income": 15000},
        (41, 45): {"mean": 6000, "std_dev": 1000, "max_income": 20000},
        (46, 50): {"mean": 7000, "std_dev": 1500, "max_income": 25000},
        (51, 55): {"mean": 8000, "std_dev": 2000, "max_income": 30000},
        (56, 60): {"mean": 9000, "std_dev": 3000, "max_income": 40000},
        (61, 65): {"mean": 10000, "std_dev": 4000, "max_income": 50000},
    }
}

# To calculate the monthly income and expenditure
def generate_income_expense(profession_undertake, current_age, num_dependents):
    # Iterate over income parameters for each profession group
    for profession_group, age_ranges in income_parameters.items():
        # Check if profession_undertake is one of the professions in the profession_group tuple
        if profession_undertake in profession_group:
            # Iterate over the age ranges and income parameters
            for age_range, params in age_ranges.items():
                if age_range[0] <= current_age <= age_range[1]:
                    mean_income = params["mean"]
                    std_dev = params["std_dev"]
                    max_income = params["max_income"]
                    unemployement_money = mean_income * 0.5  # 50% of mean income for unemployment

                    # If profession_undertake contains the word "Unemployed" before "_", return the unemployment money
                    if profession_undertake.split("_")[0] == "Unemployed":
                        income = unemployement_money
                        expenditure = generate_expenditure(income, mean_income, num_dependents) # The belong to the population under mean_income
                        return income, expenditure
                    
                    # If profession_undertake does not contain the word "Unemployed", generate income with an specific rule
                    else:
                        calc_income = int(np.random.normal(mean_income, std_dev))
                        income = max(unemployement_money, min(calc_income, max_income))  
                        expenditure = generate_expenditure(income, mean_income, num_dependents)
                        return income, expenditure

# Support function to calculate the expenditure
def generate_expenditure(income, mean_income, num_dependents):
    # Values for low and for high income people (lower possible value, mode, higher possible value) depending on the number of dependents
    low_income_params = [(0.5, 0.7, 1.5), (0.7, 0.8, 1.5), (0.8, 0.9, 1.5), (0.9, 0.9, 1.5), (0.9, 1.0, 1.5)]
    high_income_params = [(0.5, 0.6, 1.5), (0.6, 0.65, 1.5), (0.7, 0.75, 1.5), (0.7, 0.75, 1.5), (0.8, 0.85, 1.5)]
    # The parameters that should be taken depend on whether the income is below or above the mean income
    params = low_income_params if income < mean_income else high_income_params
    # Here we recover the parameters
    left, mode, right = params[min(num_dependents, 4)]  # Ensure index stays within range
    # The expenditure is calculated given a rule of min, mode, max
    expenditure = income * np.random.triangular(left=left, mode=mode, right=right)
    return expenditure



In [182]:
############################### Function to generate the credit to be requested ##################################
# Note the credits will be exactly for one year
def generate_credit_requested(income): ### It will depend on the income
    # The will maximum enter as for a credit that represent between 25% and 150% of their income and the probabilities of any values are equal, so a uniform distribution
    percentage_of_monthly_income = random.uniform(0.25, 1.5)
    yearly_income = income * 12 # must be changes once dynamically made
    credit_requested_yearly = yearly_income *percentage_of_monthly_income
    credit_requested_monthly = credit_requested_yearly / 12
    percentage_credit_month_income = credit_requested_monthly/income
    return credit_requested_monthly, percentage_credit_month_income
    

In [183]:
############################### Function to generate the whether the person defaults or not ##################################
def generate_default_label(profession, past_credits, debt_to_income_ratio_before_credit, credit_to_income_ratio):
    """This simulates a default label (0/1) based on financial risk factors."""
    """What we will use will be the """
    
    # Configurable risk settings per profession
    risk_settings = {
        "Unemployed_LowSkilled":     {"base": 0.15, "weights": (0.30, 0.6, 0.4)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "LowSkilled":                {"base": 0.08, "weights": (0.20, 0.5, 0.3)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "Unemployed_MediumSkilled": {"base": 0.12, "weights": (0.30, 0.5, 0.3)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "MediumSkilled":            {"base": 0.05, "weights": (0.20, 0.45, 0.25)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "Unemployed_HighSkilled":   {"base": 0.09, "weights": (0.30, 0.45, 0.25)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "HighSkilled":              {"base": 0.2,  "weights": (0.20, 0.4, 0.2)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
    }

    # setting variable is sett to retrieve the element of reisk_settings for the profession given
    settings = risk_settings.get(profession)
    # In case there is the profession given for the function does not match any of the professions above listed
    if not settings:
        raise ValueError(f"Unknown profession: {profession}")
    
    # Defining the weigths for the calcualtion of the probability of default
    w1, w2, w3 = settings["weights"]
    # Defining the base probability of default
    base = settings["base"]

    # Calculating the risk factor with the weigths
    risk_factor = past_credits * w1 + debt_to_income_ratio_before_credit * w2 + credit_to_income_ratio * w3
    # Defining the default probability
    default_probability = min(1, base + risk_factor)

    # We want a non-deterministic y-categorical variable that will make that same profiles will not always lead to the same result
    # So even if two people may fall on the same profile, maybe they will not default
    # return 1 if random.random() < default_probability else 0
    return int(random.random() < default_probability)

In [184]:
####################### A function to calculate the collateral ########################
def calculate_collateral(profession):
    if "HighSkilled" in profession: # If the person is HighSkilled, even if currently unemployed
        # random.random() returns a float number between 0 and 1
        if random.random() < 0.10: # 10 % of the people do not have collateral
            return 0
        else: # 90 % have a collateral between 10,000 and 50,000
            return random.randint(10000, 50000)
    elif "MediumSkilled" in profession: # If the person is HighSkilled, even if currently unemployed
        if random.random() < 0.20: # 20 % of the people do not have collateral
            return 0
        else: # 80 % have a collateral between 10,000 and 40,000
            return random.randint(10000, 40000)
    else: # If Lowskilled
        if random.random() < 0.35: # 35 % of the people do not have collateral
            return 0
        else: # 65 % have a collateral between 5,000 and 20,000
            return random.randint(5000, 20000)

In [185]:
####################### A function to estimate the seizable assets to calculate the LGD ########################
def estimate_seizable_assets(monthly_income, savings, profession, collateral):
    
    # Base asset estimation as a portion of income and savings
    # This is a proxy, people with higher income, tend to have higher assets
    # If the savinds are negative, and higher than 2 times the monthy income this will draw this to be negative
    # In the end, if no assets can be seizured, it means that maybe it is not convenient for the bank to actually give the loan
    base_asset = savings + 2 * monthly_income

    # People that are High-Skilled tend to have more assets to be seized, even if they are currently unemployed
    if "HighSkilled" in profession:
        base_asset *= 1.2
    elif "MediumSkilled" in profession:
        base_asset *= 1.0
    else:
        base_asset *= 0.8

    # If they have a collateral, more sizes are
    base_asset += collateral

    # Add some noise to simulate unpredictability
    # That means given some processes in normal life, it is not always sure that 100% of the collateral can be recovered without cost
    base_asset *= np.random.normal(1, 0.1)  # 10% variation

    # To ensure that if nothing can be seized, because in the end a negative value is returned, then we get a zero
    return max(base_asset, 0)


##### Data Generator for the original state of individuals

In [186]:
# Function to generate data accordingly to some requirements
def data_generator(number_of_customers):
    data = []
    for i in range(number_of_customers):
        
        # ---------------- X-Variables -------------------------------#
        ##### Variables not directly dependent on other variables #####
        name = f"name{i}" # names are created according to the index "i"
        age = random.randint(30, 60) # As the maximum attainable age that we want in the game is 65
        education_level = random.choices(["high school or lower", "ausbildung", "bachelor degree", "post graduate degree"],  weights=[0.3, 0.3, 0.3, 0.1], k=1)[0]
        # Number of unpaid past credits
        past_credits = random.choices([0, 1, 2, 3], weights=[0.6, 0.3, 0.08, 0.02], k=1)[0] # This emphasizes 0 and 1 unpaid credits
        # Number of dependents
        dependents = random.choices([0, 1, 2, 3, 4], weights=[0.6, 0.3, 0.06, 0.03, 0.01], k=1)[0] # This emphasizes 0 and 1 unpaid credits
        
        ##### Variables directly dependent on other variables #####
        # Generate profession based on education level
        profession = generate_profession(education_level)
        # Generate monthly income based on profession, age and number of dependents
        monthly_income = generate_income_expense(profession, age, dependents)[0]
        # Generate monthly expenditure dependening on the income, mean income and number of dependents
        monthly_expenditure = generate_income_expense(profession, age, dependents)[1]
        
        ##### Other variables generated from the variables above #####
        savings_debt = monthly_income - monthly_expenditure
        
        # Debt to income ratio: PARTIAL, before the credit
        if savings_debt < 0:
            #debt_to_income_ratio = f"{abs(savings_debt/monthly_income):.2%}"
            debt_to_income_ratio_partial = abs(savings_debt/monthly_income)
        else:
            #debt_to_income_ratio = f"{0:.2%}"
            debt_to_income_ratio_partial = 0
        
        # ---------------- Credit amount requested and time of the request--------------------------------#
        monthly_credit = generate_credit_requested(monthly_income)[0]
        credit_to_income_ratio = generate_credit_requested(monthly_income)[1]
        
        # calculating the requested duration of the loan (maybe we can make it to be then also set by the bank whether it accepts it up to this term or not)
        if 0.25 <= credit_to_income_ratio < 0.5: 
            credit_term_months = random.choices([12, 24, 36, 48, 60], weights=[0.2, 0.3, 0.2, 0.2, 0.01], k=1)[0]
        elif 0.5 <= credit_to_income_ratio < 0.75: 
            credit_term_months = random.choices([12, 24, 36, 48, 60], weights=[0.1, 0.2, 0.2, 0.2, 0.3], k=1)[0]
        elif 0.75 <= credit_to_income_ratio < 1.0: 
            credit_term_months = random.choices([12, 24, 36, 48, 60], weights=[0.08, 0.12, 0.2, 0.25, 0.35], k=1)[0]
        else:
            credit_term_months = random.choices([12, 24, 36, 48, 60], weights=[0.02, 0.08, 0.2, 0.2, 0.5], k=1)[0]
        
        # calculating the monthly debt if the monthluy credit is issued
        debt_after_credit = savings_debt - monthly_credit
        if debt_after_credit > 0: # if still the monthly savings are higher than the credit, then the total debt to incom ratio should be zero
            debt_to_income_ratio_total = 0
        else: 
            debt_to_income_ratio_total = abs(debt_after_credit/monthly_income)
            
        # collateral and seizurable asset
        collateral = calculate_collateral(profession)
        estimated_seizable_assets = estimate_seizable_assets(monthly_income, savings_debt, profession, collateral)
        
        # ---------------- Y-Variable --------------------------------#
        default_not_default = generate_default_label(profession, past_credits, debt_to_income_ratio_partial, credit_to_income_ratio)
        
        data.append({
            'name': name, # independent
            'age': age, # independent
            'educational level': education_level, # independent
            'number of not paid past credits': past_credits, # independent
            'dependents': dependents, # independent
            'profession': profession, # depends on education
            'monthly income': monthly_income, # depends on profession and age
            'monthly expenditure': monthly_expenditure, # depends on income, mean income per age and profession, and the number of dependents
            'savings (debt)': savings_debt, # monthly income - monthly expenditure
            'debt-to-income ratio before credit': debt_to_income_ratio_partial, # abs(savings_debt/monthly_income)
            'credit: monthly amount': monthly_credit, # depends on the income
            'credit-to-income ratio': credit_to_income_ratio, # credit/income
            'requested_loan_duration': credit_term_months, # depends on the credit-to-income ratio
            'debt-to-income ratio after credit': debt_to_income_ratio_total, # abs((savings_debt - credit)/monthly_income)
            'collateral': collateral, # depends on the profession
            'estimated seizable assets': estimated_seizable_assets, # it is based on the monthly income, the savings, the profession and the collateral
            'y-categorical-default': default_not_default # depending on profession, past_credits, debt_to_income_ratio_partial, credit_to_income_ratio
        })

    # Create a pandas DataFrame
    df = pd.DataFrame(data)
    return df

##### Generating one data frame

In [187]:
number_of_customers_1 = 1000
df_1 = data_generator(number_of_customers_1)
df_1

,name,age,educational level,number of not paid past credits,dependents,profession,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,requested_loan_duration,debt-to-income ratio after credit,collateral,estimated seizable assets,y-categorical-default
0,name0,55,ausbildung,2,0,MediumSkilled,4140.0,3782.323385,357.676615,0.000000,3058.176864,1.101190,60,0.652295,37500,51906.136882,1
1,name1,45,ausbildung,0,1,MediumSkilled,2822.0,3164.902732,-342.902732,0.121511,1500.863710,1.322318,60,0.653355,22204,24484.421747,0
2,name2,32,post graduate degree,0,0,HighSkilled,3398.0,2459.488749,938.511251,0.000000,4334.460458,1.191552,24,0.999396,18268,30379.326370,0
3,name3,41,high school or lower,0,0,LowSkilled,1340.0,925.813670,414.186330,0.000000,830.406060,1.190489,60,0.310612,0,2291.153957,1
4,name4,59,ausbildung,1,0,MediumSkilled,4807.0,1931.238036,2875.761964,0.000000,5920.401036,0.917852,24,0.633376,11959,27046.230490,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,41,ausbildung,0,1,MediumSkilled,2396.0,1363.744523,1032.255477,0.000000,2059.070515,0.629885,24,0.428554,15178,24325.415531,1
996,name996,42,high school or lower,0,0,LowSkilled,1599.0,1431.234645,167.765355,0.000000,2296.365128,0.594106,60,1.331207,9482,11602.979487,0
997,name997,53,ausbildung,0,0,MediumSkilled,2015.0,3346.230665,-1331.230665,0.660660,2636.267419,1.039262,48,1.968982,26495,29112.448933,1
998,name998,42,ausbildung,0,2,MediumSkilled,2338.0,2783.331523,-445.331523,0.190475,2626.523542,1.144664,48,1.313882,21149,27278.502616,1


##### DF statistics

In [188]:
df_1.describe()

,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,requested_loan_duration,debt-to-income ratio after credit,collateral,estimated seizable assets,y-categorical-default
count,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,45.117000,0.53700,0.550000,3582.585000,3317.427251,265.157749,0.119160,3180.388380,0.872606,41.928000,0.847385,18515.188000,26782.811416,0.497000
std,8.740807,0.74779,0.823516,2615.833993,2559.793748,1692.230020,0.237709,2860.835400,0.363338,15.900429,0.514609,14652.270178,19365.467544,0.500241
min,30.000000,0.00000,0.000000,500.000000,366.398867,-9566.283558,0.000000,159.662210,0.250753,12.000000,0.000000,0.000000,635.191278,0.000000
25%,38.000000,0.00000,0.000000,1714.250000,1474.382020,-307.853314,0.000000,1216.096765,0.565212,24.000000,0.457516,5867.500000,11473.679682,0.000000
50%,45.000000,0.00000,0.000000,2689.000000,2433.407978,198.122430,0.000000,2247.939165,0.871483,48.000000,0.826202,16593.000000,22752.794732,0.000000
75%,52.000000,1.00000,1.000000,4741.750000,4439.107147,804.995010,0.132215,4256.588909,1.190755,60.000000,1.187452,29952.750000,40075.014377,1.000000
max,60.000000,3.00000,4.000000,18885.000000,17569.900659,13966.626171,1.931022,22803.448902,1.498751,60.000000,3.272511,49972.000000,110841.219514,1.000000


Monthly income by profession

In [189]:
df_1.groupby('profession')['monthly income'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,345.0,6296.733333,2573.465144,1954.0,4483.00,6136.0,7610.00,18885.0
LowSkilled,286.0,1602.930070,515.007398,650.0,1188.25,1550.0,1971.25,3226.0
MediumSkilled,278.0,2815.014388,930.718464,1115.0,2052.00,2620.5,3501.75,5841.0
Unemployed_HighSkilled,35.0,3157.142857,1086.490766,1500.0,2250.00,3500.0,4000.00,4500.0
Unemployed_LowSkilled,31.0,788.709677,231.915817,500.0,550.00,750.0,1000.00,1100.0
Unemployed_MediumSkilled,25.0,1370.000000,378.043207,900.0,1150.00,1300.0,1750.00,2000.0


Debt-to-income ratio by profession

In [190]:
df_1.groupby('profession')['debt-to-income ratio before credit'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,345.0,0.135528,0.264945,0.0,0.0,0.000000,0.143784,1.931022
LowSkilled,286.0,0.116693,0.220861,0.0,0.0,0.000000,0.155707,1.289394
MediumSkilled,278.0,0.119110,0.249289,0.0,0.0,0.000000,0.114846,1.673937
Unemployed_HighSkilled,35.0,0.072798,0.098658,0.0,0.0,0.011513,0.122303,0.381672
Unemployed_LowSkilled,31.0,0.052278,0.081961,0.0,0.0,0.000000,0.093473,0.282614
Unemployed_MediumSkilled,25.0,0.069894,0.108790,0.0,0.0,0.000000,0.134635,0.327188


In [191]:
df_1.groupby('profession')['debt-to-income ratio after credit'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,345.0,0.873866,0.547918,0.000000,0.477688,0.877488,1.185374,3.272511
LowSkilled,286.0,0.832293,0.496512,0.000000,0.420332,0.801760,1.213847,2.026325
MediumSkilled,278.0,0.818019,0.522162,0.000000,0.414153,0.769741,1.172379,3.015736
Unemployed_HighSkilled,35.0,0.876131,0.386425,0.146588,0.590384,0.828750,1.201158,1.617412
Unemployed_LowSkilled,31.0,0.872769,0.447236,0.000000,0.570413,0.867070,1.204979,1.577860
Unemployed_MediumSkilled,25.0,0.909440,0.394247,0.116922,0.627510,0.850112,1.160220,1.698975


In [192]:
df_1.groupby('profession')['credit-to-income ratio'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,345.0,0.877878,0.364579,0.252870,0.568378,0.888631,1.191552,1.498089
LowSkilled,286.0,0.905152,0.364399,0.250863,0.599149,0.919115,1.216073,1.498751
MediumSkilled,278.0,0.847302,0.360564,0.250753,0.548031,0.835852,1.158817,1.484532
Unemployed_HighSkilled,35.0,0.864635,0.345520,0.270372,0.599664,0.816470,1.129822,1.474175
Unemployed_LowSkilled,31.0,0.774841,0.388301,0.266661,0.410464,0.713407,1.154674,1.415446
Unemployed_MediumSkilled,25.0,0.841300,0.348016,0.292889,0.617251,0.758921,1.062113,1.457120


In [193]:
df_1.groupby('profession')['collateral'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,345.0,26986.802899,15015.311008,0.0,16938.00,27425.0,39998.0,49972.0
LowSkilled,286.0,7762.723776,6945.544182,0.0,0.00,8295.5,13965.5,19990.0
MediumSkilled,278.0,18774.111511,12693.855393,0.0,10804.25,19522.5,29488.5,39499.0
Unemployed_HighSkilled,35.0,29196.400000,14885.399969,0.0,20947.50,32673.0,41871.5,49533.0
Unemployed_LowSkilled,31.0,7806.677419,6649.784470,0.0,0.00,8903.0,12983.5,18858.0
Unemployed_MediumSkilled,25.0,20060.720000,13261.885859,0.0,13375.00,18836.0,31016.0,39685.0


In [194]:
df_1.groupby('profession')['estimated seizable assets'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,345.0,42857.469544,18230.036885,3747.761303,29690.268626,43001.423124,55989.587786,110841.219514
LowSkilled,286.0,10413.938317,7138.641930,635.191278,3029.560095,10570.103818,16442.175199,26216.766878
MediumSkilled,278.0,24646.035514,13473.937288,1398.786650,15641.688401,25935.443913,34456.177385,57264.888275
Unemployed_HighSkilled,35.0,37337.539898,16206.978354,9796.866435,28377.392604,36766.291787,47163.586021,71023.982988
Unemployed_LowSkilled,31.0,9215.567858,6923.667108,817.048931,1589.372730,10058.795910,14763.929204,23245.259982
Unemployed_MediumSkilled,25.0,22980.147685,13336.464689,1805.451114,15871.964140,21889.980497,33791.049715,42521.812878


In [195]:
df_1.groupby('profession')['y-categorical-default'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,345.0,0.539130,0.499190,0.0,0.0,1.0,1.0,1.0
LowSkilled,286.0,0.517483,0.500570,0.0,0.0,1.0,1.0,1.0
MediumSkilled,278.0,0.406475,0.492061,0.0,0.0,0.0,1.0,1.0
Unemployed_HighSkilled,35.0,0.485714,0.507093,0.0,0.0,0.0,1.0,1.0
Unemployed_LowSkilled,31.0,0.677419,0.475191,0.0,0.0,1.0,1.0,1.0
Unemployed_MediumSkilled,25.0,0.480000,0.509902,0.0,0.0,0.0,1.0,1.0


# MODELLING PD, LGD, EAD

## PD (probability of default)
The probability that a customer with default at some point

#### Packages

In [196]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#### Converting variables into dummies

In [197]:
# Convert categorical variables to numeric
df_R = pd.get_dummies(df_1, columns=["educational level", "profession"], drop_first=True)
df_R

,name,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,...,estimated seizable assets,y-categorical-default,educational level_bachelor degree,educational level_high school or lower,educational level_post graduate degree,profession_LowSkilled,profession_MediumSkilled,profession_Unemployed_HighSkilled,profession_Unemployed_LowSkilled,profession_Unemployed_MediumSkilled
0,name0,55,2,0,4140.0,3782.323385,357.676615,0.000000,3058.176864,1.101190,...,51906.136882,1,False,False,False,False,True,False,False,False
1,name1,45,0,1,2822.0,3164.902732,-342.902732,0.121511,1500.863710,1.322318,...,24484.421747,0,False,False,False,False,True,False,False,False
2,name2,32,0,0,3398.0,2459.488749,938.511251,0.000000,4334.460458,1.191552,...,30379.326370,0,False,False,True,False,False,False,False,False
3,name3,41,0,0,1340.0,925.813670,414.186330,0.000000,830.406060,1.190489,...,2291.153957,1,False,True,False,True,False,False,False,False
4,name4,59,1,0,4807.0,1931.238036,2875.761964,0.000000,5920.401036,0.917852,...,27046.230490,1,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,41,0,1,2396.0,1363.744523,1032.255477,0.000000,2059.070515,0.629885,...,24325.415531,1,False,False,False,False,True,False,False,False
996,name996,42,0,0,1599.0,1431.234645,167.765355,0.000000,2296.365128,0.594106,...,11602.979487,0,False,True,False,True,False,False,False,False
997,name997,53,0,0,2015.0,3346.230665,-1331.230665,0.660660,2636.267419,1.039262,...,29112.448933,1,False,False,False,False,True,False,False,False
998,name998,42,0,2,2338.0,2783.331523,-445.331523,0.190475,2626.523542,1.144664,...,27278.502616,1,False,False,False,False,True,False,False,False


#### Defining X and y

In [198]:
# Column "name" is dropped from the dataframe, no need to keep it
# All the variables except y-categorical-default are X
X = df_R.drop(columns=["name", "y-categorical-default"])
y = df_R["y-categorical-default"]

#### Split between trainning and test sets

In [199]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

#### Standardize the variables for a better gradient descendt
Standardizing features to have a mean of zero ensures that all features are centered around the same baseline, which helps prevent models from being biased toward features with larger numerical values. It also makes gradient-based optimization methods like gradient descent behave more efficiently by ensuring all features contribute equally to the cost function.

In [200]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### LOGIT

##### Packages

In [201]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

##### Training the Logistic Regression

In [202]:
log_reg = LogisticRegression()
log_reg.fit(X_train_scaled, y_train)

LogisticRegression()

##### Predictions

In [203]:
y_pred = log_reg.predict(X_test_scaled)

##### Evaluations of the accuracy of model

In [204]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.71


In [205]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.69      0.76      0.73       101
           1       0.73      0.66      0.69        99

    accuracy                           0.71       200
   macro avg       0.71      0.71      0.71       200
weighted avg       0.71      0.71      0.71       200



##### Estimating the PDs

In [206]:
df_R["PD_LR"] = log_reg.predict_proba(scaler.transform(X))[:, 1]
df_R


,name,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,...,y-categorical-default,educational level_bachelor degree,educational level_high school or lower,educational level_post graduate degree,profession_LowSkilled,profession_MediumSkilled,profession_Unemployed_HighSkilled,profession_Unemployed_LowSkilled,profession_Unemployed_MediumSkilled,PD_LR
0,name0,55,2,0,4140.0,3782.323385,357.676615,0.000000,3058.176864,1.101190,...,1,False,False,False,False,True,False,False,False,0.696594
1,name1,45,0,1,2822.0,3164.902732,-342.902732,0.121511,1500.863710,1.322318,...,0,False,False,False,False,True,False,False,False,0.389355
2,name2,32,0,0,3398.0,2459.488749,938.511251,0.000000,4334.460458,1.191552,...,0,False,False,True,False,False,False,False,False,0.532231
3,name3,41,0,0,1340.0,925.813670,414.186330,0.000000,830.406060,1.190489,...,1,False,True,False,True,False,False,False,False,0.381756
4,name4,59,1,0,4807.0,1931.238036,2875.761964,0.000000,5920.401036,0.917852,...,1,False,False,False,False,True,False,False,False,0.518267
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,41,0,1,2396.0,1363.744523,1032.255477,0.000000,2059.070515,0.629885,...,1,False,False,False,False,True,False,False,False,0.200728
996,name996,42,0,0,1599.0,1431.234645,167.765355,0.000000,2296.365128,0.594106,...,0,False,True,False,True,False,False,False,False,0.186393
997,name997,53,0,0,2015.0,3346.230665,-1331.230665,0.660660,2636.267419,1.039262,...,1,False,False,False,False,True,False,False,False,0.665779
998,name998,42,0,2,2338.0,2783.331523,-445.331523,0.190475,2626.523542,1.144664,...,1,False,False,False,False,True,False,False,False,0.392400


### Using Neuronal Networks

##### Packages

In [207]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

##### Building the Neuronal Network

In [208]:
# We may try out:

# tanh: The hyperbolic tangent function outputs values between -1 and 1, making it useful for hidden layers 
# where you want activations that are zero-centered, which helps in faster convergence and avoids saturation for small inputs.
 
# relu: The Rectified Linear Unit activation function outputs zero for any negative input and passes positive values as they are.
# It is widely used in hidden layers for its simplicity and effectiveness, and helps avoid the vanishing gradient problem seen with functions like sigmoid and tanh.

# softmax: Softmax is typically used in the output layer for multi-class classification tasks. It converts the raw outputs into probabilities, 
# ensuring that the sum of all output values equals 1, representing the probability distribution over multiple classes.

# This is like having an input which is your variable X, then 32 neurons process the input features,
# using a function (in this case "tanh") to calculate the weights and transformations at each neuron. 
# The results are then passed to a subsequent layer with 16 neurons, where again "tanh" is applied to further transform the data.
# In the end, everything is passed through a final neuron that uses a "sigmoid" (logistic) function 
# to produce an output between [0, 1], representing a probability for binary classification.
model = Sequential([  # Each layer is run after the other, forming a linear stack of layers.
    # The first Dense layer applies 32 units (neurons) and uses the "tanh" activation function.
    # The input_shape corresponds to the number of features in the dataset (X_train_scaled).
    Dense(32, activation='tanh', input_shape=(X_train_scaled.shape[1],)),
    
    # The second Dense layer applies 16 units (neurons) and uses "tanh" activation function.
    # "tanh" ensures that the output of each neuron will be between -1 and 1, centering the activations.
    Dense(16, activation='tanh'),
    
    # The final Dense layer outputs a single value, which is the probability of the positive class.
    # Sigmoid activation squashes the output to a value between 0 and 1.
    # This is commonly used for binary classification, where the output is a probability of class 1.
    Dense(1, activation='sigmoid')
])

/Users/bonjour/opt/anaconda3/envs/bankgame/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


##### Compiling the model

In [209]:
# The model is being compiled with the following parameters:
# optimizer='adam': The Adam optimizer is being used. It is an adaptive learning rate optimization algorithm that 
#  combines the benefits of both AdaGrad and RMSProp, making it well-suited for most deep learning models.
# loss='binary_crossentropy': The loss function used is binary cross-entropy, which is appropriate for binary classification 
#  tasks where the output is a probability of belonging to one of two classes, which is the case of our y-variable
# metrics=['accuracy']: The model will track accuracy as the evaluation metric during training and testing, 
#  which measures the percentage of correct predictions.

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

##### Training the model

In [210]:
# The model is fit with X_train_scaled: This means the model is being trained on the scaled training data (X_train_scaled) 
# using the corresponding labels (y_train).

# Uses 25 epochs: An epoch refers to one full pass through the entire training dataset. 
# The model will train for 25 epochs, meaning it will go through the data 25 times to learn the optimal weights.

# It will use a batch size of 32: The model will train using 32 samples (or rows of data) at a time, and after processing 
# those 32, it updates the weights before moving on to the next 32 samples. 

# During training, the model's performance is periodically evaluated on the validation set (X_test_scaled and y_test) 
# to monitor overfitting and to adjust the training accordingly.

model_NN = model.fit(X_train_scaled, y_train, epochs=25, batch_size=32, validation_data=(X_test_scaled, y_test))

Epoch 1/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.4306 - loss: 0.7721 - val_accuracy: 0.4900 - val_loss: 0.7167
Epoch 2/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5364 - loss: 0.6958 - val_accuracy: 0.5950 - val_loss: 0.6766
Epoch 3/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6500 - loss: 0.6488 - val_accuracy: 0.6050 - val_loss: 0.6550
Epoch 4/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6523 - loss: 0.6313 - val_accuracy: 0.6400 - val_loss: 0.6372
Epoch 5/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6535 - loss: 0.6292 - val_accuracy: 0.6400 - val_loss: 0.6270
Epoch 6/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6853 - loss: 0.6000 - val_accuracy: 0.6250 - val_loss: 0.6217
Epoch 7/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7075 - loss: 0.5787 - val_accuracy: 0.6550 - val_loss: 0.6090
Epoch 8/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7009 - loss: 0.5757 - val_accuracy: 0.6450 - val_los

##### We want to get the last accuracy of the last epoch and the minimal and highest accuracy values for comparison

In [211]:
train_accuracies_NN = model_NN.history['accuracy']

In [212]:
# final accuracy value
final_accuracy = train_accuracies_NN[-1]
# maximal accuracy value
min_accuracy = min(train_accuracies_NN)
# minimal accuracy value
max_accuracy = max(train_accuracies_NN)
# average accuracy value
average_accuracy = sum(train_accuracies_NN) / len(train_accuracies_NN)

In [213]:
# Print the results
print(f'Final accuracy: {final_accuracy:.4f}')
print(f'Minimum accuracy during training of the NN: {min_accuracy:.4f}')
print(f'Maximum accuracy during training of the NN: {max_accuracy:.4f}')
print(f'Average accuracy during training of the NN: {average_accuracy:.4f}')

Final accuracy: 0.7225
Minimum accuracy during training of the NN: 0.4625
Maximum accuracy during training of the NN: 0.7225
Average accuracy during training of the NN: 0.6794


##### Estimating the PDs

In [214]:
df_R["PD_NN"]= model.predict(scaler.transform(X))
df_R

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


,name,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,...,educational level_bachelor degree,educational level_high school or lower,educational level_post graduate degree,profession_LowSkilled,profession_MediumSkilled,profession_Unemployed_HighSkilled,profession_Unemployed_LowSkilled,profession_Unemployed_MediumSkilled,PD_LR,PD_NN
0,name0,55,2,0,4140.0,3782.323385,357.676615,0.000000,3058.176864,1.101190,...,False,False,False,False,True,False,False,False,0.696594,0.731952
1,name1,45,0,1,2822.0,3164.902732,-342.902732,0.121511,1500.863710,1.322318,...,False,False,False,False,True,False,False,False,0.389355,0.354226
2,name2,32,0,0,3398.0,2459.488749,938.511251,0.000000,4334.460458,1.191552,...,False,False,True,False,False,False,False,False,0.532231,0.583837
3,name3,41,0,0,1340.0,925.813670,414.186330,0.000000,830.406060,1.190489,...,False,True,False,True,False,False,False,False,0.381756,0.354077
4,name4,59,1,0,4807.0,1931.238036,2875.761964,0.000000,5920.401036,0.917852,...,False,False,False,False,True,False,False,False,0.518267,0.435634
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,41,0,1,2396.0,1363.744523,1032.255477,0.000000,2059.070515,0.629885,...,False,False,False,False,True,False,False,False,0.200728,0.154052
996,name996,42,0,0,1599.0,1431.234645,167.765355,0.000000,2296.365128,0.594106,...,False,True,False,True,False,False,False,False,0.186393,0.114753
997,name997,53,0,0,2015.0,3346.230665,-1331.230665,0.660660,2636.267419,1.039262,...,False,False,False,False,True,False,False,False,0.665779,0.785036
998,name998,42,0,2,2338.0,2783.331523,-445.331523,0.190475,2626.523542,1.144664,...,False,False,False,False,True,False,False,False,0.392400,0.425742


### Using Random Forests

##### Packages

In [215]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

##### Fitting the model

In [216]:
# Initialize the RandomForestClassifier with the following parameters:
# n_estimators=100: This sets the number of decision trees (estimators) in the forest. The model will train 100 individual trees and aggregate their results to make predictions.
# random_state=42: This ensures reproducibility by fixing the random seed used in the training process. 
# To get the same result even if the code is run multiple times (obviously this will only be affected by the random nature of our data)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

RandomForestClassifier(random_state=42)

##### Accuracy of the model

In [217]:
# Predict class labels for X_test_scaled
y_pred = rf_model.predict(X_test_scaled)
# Calculate accuracy by comparing the predicted labels with our simulated data y
accuracy = accuracy_score(y_test, y_pred)

print(f'Accuracy: {accuracy:.4f}')

Accuracy: 0.6100


##### Estimating the PDs

In [218]:
# df_R = df_R.iloc[:len(X_test_scaled)]
# df_R["PD_RF"] = rf_model.predict_proba(X_test_scaled)[:, 1]
df_R["PD_RF"] = rf_model.predict_proba(scaler.transform(X))[:, 1] # needs to be checked
df_R

,name,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,...,educational level_high school or lower,educational level_post graduate degree,profession_LowSkilled,profession_MediumSkilled,profession_Unemployed_HighSkilled,profession_Unemployed_LowSkilled,profession_Unemployed_MediumSkilled,PD_LR,PD_NN,PD_RF
0,name0,55,2,0,4140.0,3782.323385,357.676615,0.000000,3058.176864,1.101190,...,False,False,False,True,False,False,False,0.696594,0.731952,0.93
1,name1,45,0,1,2822.0,3164.902732,-342.902732,0.121511,1500.863710,1.322318,...,False,False,False,True,False,False,False,0.389355,0.354226,0.17
2,name2,32,0,0,3398.0,2459.488749,938.511251,0.000000,4334.460458,1.191552,...,False,True,False,False,False,False,False,0.532231,0.583837,0.69
3,name3,41,0,0,1340.0,925.813670,414.186330,0.000000,830.406060,1.190489,...,True,False,True,False,False,False,False,0.381756,0.354077,0.76
4,name4,59,1,0,4807.0,1931.238036,2875.761964,0.000000,5920.401036,0.917852,...,False,False,False,True,False,False,False,0.518267,0.435634,0.86
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,41,0,1,2396.0,1363.744523,1032.255477,0.000000,2059.070515,0.629885,...,False,False,False,True,False,False,False,0.200728,0.154052,0.13
996,name996,42,0,0,1599.0,1431.234645,167.765355,0.000000,2296.365128,0.594106,...,True,False,True,False,False,False,False,0.186393,0.114753,0.01
997,name997,53,0,0,2015.0,3346.230665,-1331.230665,0.660660,2636.267419,1.039262,...,False,False,False,True,False,False,False,0.665779,0.785036,0.91
998,name998,42,0,2,2338.0,2783.331523,-445.331523,0.190475,2626.523542,1.144664,...,False,False,False,True,False,False,False,0.392400,0.425742,0.83


In [219]:
df_R.describe()

,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,requested_loan_duration,debt-to-income ratio after credit,collateral,estimated seizable assets,y-categorical-default,PD_LR,PD_NN,PD_RF
count,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,45.117000,0.53700,0.550000,3582.585000,3317.427251,265.157749,0.119160,3180.388380,0.872606,41.928000,0.847385,18515.188000,26782.811416,0.497000,0.497001,0.503954,0.495570
std,8.740807,0.74779,0.823516,2615.833993,2559.793748,1692.230020,0.237709,2860.835400,0.363338,15.900429,0.514609,14652.270178,19365.467544,0.500241,0.217170,0.226343,0.319322
min,30.000000,0.00000,0.000000,500.000000,366.398867,-9566.283558,0.000000,159.662210,0.250753,12.000000,0.000000,0.000000,635.191278,0.000000,0.097214,0.080272,0.010000
25%,38.000000,0.00000,0.000000,1714.250000,1474.382020,-307.853314,0.000000,1216.096765,0.565212,24.000000,0.457516,5867.500000,11473.679682,0.000000,0.326317,0.303440,0.170000
50%,45.000000,0.00000,0.000000,2689.000000,2433.407978,198.122430,0.000000,2247.939165,0.871483,48.000000,0.826202,16593.000000,22752.794732,0.000000,0.477011,0.501711,0.490000
75%,52.000000,1.00000,1.000000,4741.750000,4439.107147,804.995010,0.132215,4256.588909,1.190755,60.000000,1.187452,29952.750000,40075.014377,1.000000,0.652775,0.692430,0.810000
max,60.000000,3.00000,4.000000,18885.000000,17569.900659,13966.626171,1.931022,22803.448902,1.498751,60.000000,3.272511,49972.000000,110841.219514,1.000000,0.992247,0.953589,0.990000


## EAD (exposure at default)
It is the total amount at risk at the moment the borrower defaults

That is the amount fo the credit that has been unpaid at the moment of the calculation

In [220]:
df_R["EAD"] = df_R["credit: monthly amount"] * 12
df_R

,name,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,...,educational level_post graduate degree,profession_LowSkilled,profession_MediumSkilled,profession_Unemployed_HighSkilled,profession_Unemployed_LowSkilled,profession_Unemployed_MediumSkilled,PD_LR,PD_NN,PD_RF,EAD
0,name0,55,2,0,4140.0,3782.323385,357.676615,0.000000,3058.176864,1.101190,...,False,False,True,False,False,False,0.696594,0.731952,0.93,36698.122369
1,name1,45,0,1,2822.0,3164.902732,-342.902732,0.121511,1500.863710,1.322318,...,False,False,True,False,False,False,0.389355,0.354226,0.17,18010.364515
2,name2,32,0,0,3398.0,2459.488749,938.511251,0.000000,4334.460458,1.191552,...,True,False,False,False,False,False,0.532231,0.583837,0.69,52013.525496
3,name3,41,0,0,1340.0,925.813670,414.186330,0.000000,830.406060,1.190489,...,False,True,False,False,False,False,0.381756,0.354077,0.76,9964.872725
4,name4,59,1,0,4807.0,1931.238036,2875.761964,0.000000,5920.401036,0.917852,...,False,False,True,False,False,False,0.518267,0.435634,0.86,71044.812429
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,41,0,1,2396.0,1363.744523,1032.255477,0.000000,2059.070515,0.629885,...,False,False,True,False,False,False,0.200728,0.154052,0.13,24708.846177
996,name996,42,0,0,1599.0,1431.234645,167.765355,0.000000,2296.365128,0.594106,...,False,True,False,False,False,False,0.186393,0.114753,0.01,27556.381539
997,name997,53,0,0,2015.0,3346.230665,-1331.230665,0.660660,2636.267419,1.039262,...,False,False,True,False,False,False,0.665779,0.785036,0.91,31635.209032
998,name998,42,0,2,2338.0,2783.331523,-445.331523,0.190475,2626.523542,1.144664,...,False,False,True,False,False,False,0.392400,0.425742,0.83,31518.282509


## LGD (loss given default)

Example:

1. A borrower takes a loan of €10,000. 
2. They default after paying back €2,000, and the bank recovers €4,000 by seizing assets. 

That means:

Total Recovered = €2,000 (paid) + €4,000 (recovered from assets) = €6,000

Total Loss = €10,000 - €6,000 = €4,000

LGD = €4,000 (Total Loss)/ €10,000 (Loan Amount) = 40%



#### In our approximation 
LGD = (EAD - what can be sized)/EAD = 1 - (What can be seized/EAD)

In [221]:
def estimate_lgd(EAD, seizable_assets):
    lgd = 1 - (seizable_assets / EAD)
    return max(0, min(lgd, 1))  # The lgd should be between 0 and 1

In [222]:
df_R["LGD"] = df_R.apply(lambda row: estimate_lgd(row["EAD"], row["estimated seizable assets"]), axis=1) # This should be applied row by row

In [223]:
df_R.describe()

,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,requested_loan_duration,debt-to-income ratio after credit,collateral,estimated seizable assets,y-categorical-default,PD_LR,PD_NN,PD_RF,EAD,LGD
count,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,45.117000,0.53700,0.550000,3582.585000,3317.427251,265.157749,0.119160,3180.388380,0.872606,41.928000,0.847385,18515.188000,26782.811416,0.497000,0.497001,0.503954,0.495570,38164.660559,0.332070
std,8.740807,0.74779,0.823516,2615.833993,2559.793748,1692.230020,0.237709,2860.835400,0.363338,15.900429,0.514609,14652.270178,19365.467544,0.500241,0.217170,0.226343,0.319322,34330.024805,0.325800
min,30.000000,0.00000,0.000000,500.000000,366.398867,-9566.283558,0.000000,159.662210,0.250753,12.000000,0.000000,0.000000,635.191278,0.000000,0.097214,0.080272,0.010000,1915.946520,0.000000
25%,38.000000,0.00000,0.000000,1714.250000,1474.382020,-307.853314,0.000000,1216.096765,0.565212,24.000000,0.457516,5867.500000,11473.679682,0.000000,0.326317,0.303440,0.170000,14593.161176,0.000000
50%,45.000000,0.00000,0.000000,2689.000000,2433.407978,198.122430,0.000000,2247.939165,0.871483,48.000000,0.826202,16593.000000,22752.794732,0.000000,0.477011,0.501711,0.490000,26975.269977,0.261933
75%,52.000000,1.00000,1.000000,4741.750000,4439.107147,804.995010,0.132215,4256.588909,1.190755,60.000000,1.187452,29952.750000,40075.014377,1.000000,0.652775,0.692430,0.810000,51079.066905,0.605880
max,60.000000,3.00000,4.000000,18885.000000,17569.900659,13966.626171,1.931022,22803.448902,1.498751,60.000000,3.272511,49972.000000,110841.219514,1.000000,0.992247,0.953589,0.990000,273641.386818,0.934352
